# 04 — Modeling (optional)
**Target:** `log_pay`. **Predictors:** experience, role, remote status, education, org size.
No other compensation fields are used as predictors (leakage).
All preprocessing is fit on the training set only, inside a Pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
df = pd.read_csv("../data/processed/us_fulltime_clean.csv")
print(df.shape)
df.head()

In [ ]:
num = ["WorkExp_num"]
cat = ["role", "RemoteWork", "EdLevel", "OrgSize"]
X, y = df[num + cat], df["log_pay"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat),
])

models = {
    "Baseline (median)": DummyRegressor(strategy="median"),
    "Linear regression": LinearRegression(),
    "KNN regression (k=15)": KNeighborsRegressor(n_neighbors=15),
    "Regression tree (depth=5)": DecisionTreeRegressor(max_depth=5, random_state=42),
}

In [ ]:
rows = []
for name, model in models.items():
    pipe = Pipeline([("prep", prep), ("model", model)]).fit(X_train, y_train)
    pred_dollars = np.exp(pipe.predict(X_test))
    true_dollars = np.exp(y_test)
    rows.append({"Model": name,
                 "MAE ($)": mean_absolute_error(true_dollars, pred_dollars),
                 "RMSE ($)": mean_squared_error(true_dollars, pred_dollars) ** 0.5})
pd.DataFrame(rows).round(0)

**TODO:** Explain how each model works in plain terms, which performed best, and whether any beat the baseline meaningfully. Any tuning must use the training set only (e.g., cross-validation), never the test set.